<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: Bayesian workflow

**Short Bayesian course — worked example**

$$
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit}
\rightarrow
LD50
\rightarrow
\text{posterior predictive}
$$

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

# Figures used in the slides: Slides/figures/<notebook>_<section>[_<n>].svg
NOTEBOOK = "bioassay"
FIG_DIR = Path("../Slides/figures") if Path("../Slides").is_dir() else Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_slide_figure(fig, section, number=None):
    """Save a Matplotlib figure or ArviZ PlotCollection for the slides."""
    suffix = f"_{number}" if number else ""
    fig.savefig(
        FIG_DIR / f"{NOTEBOOK}_{section}{suffix}.svg", bbox_inches="tight", transparent=True
    )

def add_interval_legend(ax, line_label="median", observed_label="observed"):
    """Compact legend for the 50%/90% intervals, median, and observations."""
    ax.legend(
        handles=[
            Line2D([0], [0], color="C1", lw=1.6, label=line_label),
            Patch(facecolor="C0", alpha=0.9, label="50% HDI"),
            Patch(facecolor="C0", alpha=0.28, label="90% HDI"),
            Line2D(
                [0], [0], marker="o", linestyle="none", color="black",
                markersize=4.5, label=observed_label,
            ),
        ],
        loc="upper left",
        ncols=2,
        fontsize=8,
        handlelength=1.4,
        handletextpad=0.45,
        columnspacing=0.9,
        borderpad=0.35,
        labelspacing=0.35,
        frameon=True,
        framealpha=0.88,
    )

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(dose, deaths / n, s=70, color="black")
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
save_slide_figure(fig, "data")

# LD50 is the dose at which the probability of death crosses 50%.
ax.axhline(0.5, color="C1", linestyle="--")
save_slide_figure(fig, "data", 2)
plt.show()

## 2. Model

$$
y_j\sim\operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j
$$

$$
\alpha\sim N(0,5),
\qquad
\beta\sim\operatorname{HalfNormal}(5).
$$

In [ ]:
coords = {"dose_log_g_ml": dose}

with pm.Model(coords=coords) as model:
    dose_data = pm.Data("dose", dose, dims="dose_log_g_ml")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose_data
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_log_g_ml",
    )

    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n,
        logit_p=logit_p,
        observed=deaths,
        dims="dose_log_g_ml",
    )

## 3. Prior predictive

**Question:** Before fitting, what death counts do these priors say are plausible?

These intervals summarize replicated **death counts** $y$. Because $y$ is discrete, an HDI need not be centered on its median; for a skewed Binomial predictive distribution the median can lie on an HDI boundary. Dose is continuous, so the bands run across dose even though only four doses were tested.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["alpha", "beta", "p", "deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    prior,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="prior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    figure_kwargs={"figsize": (5, 3.4)},
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median", observed_label="observed")
save_slide_figure(plt.gcf(), "prior-predictive")
plt.show()

## 4. Fit and diagnose

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
pc = azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
    figure_kwargs={"figsize": (8, 4)},
)
save_slide_figure(pc, "fit-and-diagnose")

## 5. Posterior dose-response fit

This plot shows uncertainty about the underlying mean response, $5p$. The 50% and 90% HDIs are intervals for the fitted dose-response relationship; they do **not** yet include the extra Binomial variation in a new group of five animals.

In [ ]:
idata["posterior"]["expected_deaths"] = 5 * idata["posterior"]["p"]

azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="posterior",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    figure_kwargs={"figsize": (5, 3.4)},
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median 5p", observed_label="observed")
save_slide_figure(plt.gcf(), "posterior-dose-response-fit")
plt.show()

For comparison, the same plot before seeing the data: the dose-response curves the prior allows.

In [ ]:
prior["prior"]["expected_deaths"] = 5 * prior["prior"]["p"]

azp.plot_lm(
    prior,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="prior",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    figure_kwargs={"figsize": (5, 3.4)},
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median 5p", observed_label="observed")
save_slide_figure(plt.gcf(), "posterior-dose-response-fit", 2)
plt.show()

## 6. LD50

$$
LD50=-\frac{\alpha}{\beta}.
$$

It is stored as a PyMC deterministic quantity.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
pc = azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
    figure_kwargs={"figsize": (5, 3)},
)
save_slide_figure(pc, "ld50")

## 7. Posterior predictive

**Question:** After fitting, can the model generate death counts like those observed?

This uses the same display as the prior predictive check, but now the replicated counts are conditioned on the data. Unlike the posterior dose-response plot above, these intervals are for $y$, so they include both uncertainty about $p$ and Binomial outcome variability.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="posterior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    figure_kwargs={"figsize": (5, 3.4)},
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median", observed_label="observed")
save_slide_figure(plt.gcf(), "posterior-predictive")
plt.show()

## Explore

- Remove the positive-slope constraint.
- Change the priors and rerun the prior predictive check.

Source: Gelman & Vehtari, *Bayesian Workflow*, §3.5.